<a href="https://colab.research.google.com/github/erickcorrea3092/Cheminformatics-course/blob/main/4_6_molplotly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Molplotly

molplotly es un complemento para la biblioteca de visualización de datos Plotly en Python, construido sobre RDKit, que permite mostrar imágenes 2D de moléculas directamente en gráficos interactivos (como diagramas de dispersión) cuando pasas el cursor sobre los puntos de datos, facilitando la quimioinformática y visualización de datos químicos interactivos.

In [1]:
from IPython.utils import io
import tqdm.notebook
import os, os.path, sys, random, subprocess
total = 100
a= True
if a== True:
  with tqdm.notebook.tqdm(total=total) as pbar:
    with io.capture_output() as captured:
      pbar.update(0)
      ! pip install dash==3.2.0
      pbar.update(33)
      ! pip install rdkit
      pbar.update(33)
      ! pip install molplotly
      pbar.update(34)

  0%|          | 0/100 [00:00<?, ?it/s]

In [2]:
import pandas as pd
import plotly.express as px
import molplotly

Carguemos el conjunto de datos ESOL de [ESOL: Estimación de la solubilidad acuosa directamente a partir de la estructura molecular](https://doi.org/10.1021/ci034243x), alojado amablemente por el equipo de [deepchem](https://github.com/deepchem/deepchem), pero también incluido como `example.csv` en el repositorio.

In [3]:
# df_esol = pd.read_csv('example.csv')
df_esol = pd.read_csv('https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv')
df_esol['y_pred'] = df_esol['ESOL predicted log solubility in mols per litre']
df_esol['y_true'] = df_esol['measured log solubility in mols per litre']

In [4]:
df_esol.head()

,Compound ID,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre,smiles,y_pred,y_true
0,Amigdalin,-0.974,1,457.432,7,3,7,202.32,-0.77,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...,-0.974,-0.77
1,Fenfuram,-2.885,1,201.225,1,2,2,42.24,-3.30,Cc1occc1C(=O)Nc2ccccc2,-2.885,-3.30
2,citral,-2.579,1,152.237,0,0,4,17.07,-2.06,CC(C)=CCCC(C)=CC(=O),-2.579,-2.06
3,Picene,-6.618,2,278.354,0,5,0,0.00,-7.87,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43,-6.618,-7.87
4,Thiophene,-2.232,2,84.143,0,1,0,0.00,-1.33,c1ccsc1,-2.232,-1.33


In [5]:
# Veamos los compuestos incluidos
print(df_esol['Compound ID'].tolist())

['Amigdalin', 'Fenfuram', 'citral', 'Picene', 'Thiophene', 'benzothiazole', "2,2,4,6,6'-PCB", 'Estradiol', 'Dieldrin', 'Rotenone', '2-pyrrolidone', '2-Chloronapthalene', '1-Pentene ', 'Primidone', 'Tetradecane', '2-Chloropropane', '2-Methylbutanol', 'Benzonitrile', 'Diazinon', '2-Undecanol', "2,2',3,4,6-PCB", 'Lenacil', 'Phorate', 'Phenacetin', 'Dinitramine', '1-Heptanol', 'Theophylline', 'Butethal', "P,P'-DDE", 'Methyl octanoate', '1,4-Diethylbenzene ', 'Terbufos', 'Phenmedipham', '1,1-Dichloroethylene', '1-Methylfluorene', 'Valeraldehyde', 'Diphenylamine', 'Fenothiocarb', 'Piperophos', '1-Iodoheptane', '3-Chlorobiphenyl', '4-Pentene-1-ol', 'Cyclobutyl-5-spirobarbituric acid', 'menthol', 'Isopropyl formate', '2-Heptanol ', 'p-Bromoacetanilide', 'brompyrazone', 'nifedipine', '2,7-dimethylquinoline', '1-Octyne ', 'cyclobarbital', 'Chrysene', 'Bromacil', "2,2',3,3',5,6-PCB", '2-Methylphenol', '2,2,5-Trimethylhexane', '1,4-Dimethylnaphthalene ', '6-Methylchrysene', '2-Pentanone', "2,2',3,

## Gráfica de dispersión simple

Hagamos un diagrama de dispersión que compare las solubilidades medidas y predichas usando [`plotly`](https://plotly.com/python/)

In [6]:
df_esol['delY'] = df_esol["y_pred"] - df_esol["y_true"]

fig_scatter = px.scatter(df_esol,
                         x="y_true",
                         y="y_pred",
                         color='delY',
                         title='ESOL Regression (default plotly)',
                         labels={'y_pred': 'Predicted Solubility',
                                 'y_true': 'Measured Solubility',
                                 'delY': 'ΔY'},
                         width=1200,
                         height=800)

# This adds a dashed line for what a perfect model _should_ predict
y = df_esol["y_true"].values
fig_scatter.add_shape(
    type="line", line=dict(dash='dash'),
    x0=y.min(), y0=y.min(),
    x1=y.max(), y1=y.max()
)

fig_scatter.show()

¡Ahora todo lo que tenemos que hacer es agregar moléculas `add_molecules`!

In [7]:
fig_scatter.update_layout(title='ESOL Regression (with add_molecules!)')

app_scatter = molplotly.add_molecules(fig=fig_scatter,
                                      df=df_esol,
                                      smiles_col='smiles',
                                    #  title_col='Compound ID'
                                      )

# change the arguments here to run the dash app on an external server and/or change the size of the app!
app_scatter.run(port=8700, #height=1000
                )


/usr/local/lib/python3.13/dist-packages/dash/dash.py:634: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.



<IPython.core.display.Javascript object>

## Formatting

Genial, ¿verdad? Exploremos algunas opciones de formato:


# Transparencia del cuadro flotante

la transparencia del hoverbox y la molécula dibujada se pueden controlar mediante los parámetros alfa y mol_alpha, respectivamente. Los valores predeterminados son 0,75 y 0,7 por preferencia personal (esos son los valores en el gráfico anterior), aquí hay un ejemplo con valores alfa más pequeños:

In [8]:
fig_scatter.update_layout(title='ESOL Regression with more transparent hoverboxes')

app_scatter_alpha = molplotly.add_molecules(fig=fig_scatter,
                                      df=df_esol,
                                      smiles_col='smiles',
                                      #title_col='Compound ID',
                                      alpha=0.7,
                                      mol_alpha=0.10,
                                      )

# change the arguments here to run the dash app on an external server and/or change the size of the app!
app_scatter_alpha.run(mode='inline', port=8800, height=1000)


<IPython.core.display.Javascript object>

# Subtítulos adicionales

Además de mostrar las coordenadas (x,y) (puede desactivarlas usando `show_coords=False`), podemos agregar valores adicionales para que aparezcan en la información sobre herramientas del mouse especificando `caption_cols`; los valores en estas columnas de df_esol también se muestran en el cuadro flotante.

También podemos aplicar algunas transformaciones de funciones a los títulos a través de `caption_transform`; en este ejemplo, redondeamos todos nuestros números a 2 decimales.



In [9]:
fig_scatter.update_layout(
    title='ESOL Regression (with add_molecules & extra captions)')

app_scatter_with_captions = molplotly.add_molecules(fig=fig_scatter,
                                                    df=df_esol,
                                                    smiles_col='smiles',
                                                   # title_col='Compound ID',
                                                    caption_cols=['Molecular Weight', 'Number of Rings','Number of H-Bond Donors',
                                                                  'Number of Rotatable Bonds', 'Polar Surface Area'],
                                                    caption_transform={'Predicted Solubility': lambda x: f"{x:.2f}",
                                                                       'Measured Solubility': lambda x: f"{x:.2f}",
                                                                       'Molecular Weight': lambda x: f"{x:.2f}"
                                                                       },
                                                    show_coords=True)

app_scatter_with_captions.run(mode='inline', port=8702, height=1000)


<IPython.core.display.Javascript object>

### Colores y tamaño

¿Qué hay de agregar colores? Aquí hice una división aleatoria arbitraria del conjunto de datos en "entrenamiento" y "prueba". Al trazar, esto conduce a dos "curvas" de trazado separadas, por lo que la condición que determina el color de los puntos debe pasarse a la función `add_molecules` para que se seleccionen los SMILES correctas para la visualización; esto se hace a través de `color_col`. Observe que el "título" de las moléculas en el cuadro flotante tiene el mismo color que el punto de datos.

Por diversión, también utilicé el argumento "tamaño" `size` en el diagrama de dispersión para cambiar el tamaño de los marcadores en proporción al peso molecular.

(Observe que he estado eligiendo diferentes números de "puerto" `port` en todos mis gráficos, ¡esto es para que no interfieran entre sí!)

In [10]:
from sklearn.model_selection import train_test_split

train_inds, test_inds = train_test_split(df_esol.index)
df_esol['dataset'] = [
    'Train' if x in train_inds else 'Test' for x in df_esol.index]

fig_train_test = px.scatter(df_esol,
                            x="y_true",
                            y="y_pred",
                            size='Molecular Weight',
                            color='dataset',
                            title='ESOL Regression (colored by random train/test split)',
                            labels={'y_pred': 'Predicted Solubility',
                                    'y_true': 'Measured Solubility'},
                            width=1200,
                            height=800)
# fig.show()
app_train_test = molplotly.add_molecules(fig=fig_train_test,
                                         df=df_esol,
                                         smiles_col='smiles',
                                        #  title_col='Compound ID',
                                         color_col='dataset')

app_train_test.run(port=8703, height=1000)


/usr/local/lib/python3.13/dist-packages/dash/dash.py:634: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.



<IPython.core.display.Javascript object>

# Otros Ejemplos

## Gráficas de Dispersión 3D

La configuración de coordenadas predeterminada no funciona con diagramas de dispersión 3D, por lo que deberá configurar `show_coords=False` e ingresar manualmente las coordenadas en `caption_cols`.

In [11]:
fig_3d = px.scatter_3d(df_esol,
                       x="y_pred",
                       y="y_true",
                       z="Molecular Weight",
                       width=1000,
                       height=800)

app_3d = molplotly.add_molecules(
    fig=fig_3d,
    df=df_esol,
    smiles_col="smiles",
    caption_cols=["y_pred", "y_true", "Molecular Weight"],
    show_coords=False,
)

# Use molplotly's utility function for inline display in Colab
app_3d.run(port=8704, height=850)

/usr/local/lib/python3.13/dist-packages/dash/dash.py:634: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.



<IPython.core.display.Javascript object>

## Matrices de dispersión

Para visualizar la relación entre múltiples variables a la vez, use una matriz de diagramas de dispersión.

Aquí aumenté el ancho del cuadro flotante usando el parámetro 'ancho' `width` porque los títulos de los subtítulos se estaban volviendo largos; también he usado `show_coords=False` porque las coordenadas $(x, y)$ para diagramas de dispersión no triviales se vuelven confusas.

In [12]:
features = ['Number of H-Bond Donors',
            'Number of Rings',
            'Number of Rotatable Bonds',
            'Polar Surface Area']
fig_matrix = px.scatter_matrix(df_esol,
                               dimensions=features,
                               width=1200,
                               height=800,
                               title='Scatter matrix of molecular properties')

# Only show informative lower triangle
fig_matrix.update_traces(diagonal_visible=False, showupperhalf=False)


app_matrix = molplotly.add_molecules(fig=fig_matrix,
                                     df=df_esol,
                                     smiles_col='smiles',
                                    #  title_col='Compound ID',
                                     caption_cols=features,
                                     width=200,
                                     show_coords=False)


app_matrix.run(#mode='inline',
               port=8706, height=1000)

/usr/local/lib/python3.13/dist-packages/dash/dash.py:634: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.



<IPython.core.display.Javascript object>

Visualización de componentes PCA de MorganFP

Una forma común de visualizar un conjunto de datos moleculares es calcular las huellas de Morgan de las moléculas y visualizarlas en una incrustación 2D (por ejemplo, PCA/t-SNE). En este ejemplo, trazaré los 2 componentes PCA más grandes para ESOL e inspeccionaré los datos.

¡Primero calculemos los componentes PCA!

In [13]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from sklearn.decomposition import PCA


def smi_to_fp(smi):
    fp = AllChem.GetMorganFingerprintAsBitVect(
        Chem.MolFromSmiles(smi), 2, nBits=1024)
    arr = np.zeros((0,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

esol_fps = np.array([smi_to_fp(smi) for smi in df_esol['smiles']])
pca = PCA(n_components=2)
components = pca.fit_transform(esol_fps.reshape(-1, 1024))
df_esol['PCA-1'] = components[:, 0]
df_esol['PCA-2'] = components[:, 1]


[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerator
[14:38:06] DEPRECATION WARNING: please use MorganGenerat

¡Y ahora veámoslos!

Con "molplotly", es muy fácil ver qué moléculas están dónde: moléculas de esteroides en la parte superior, alcoholes en la parte inferior izquierda, compuestos aromáticos clorados en la parte inferior derecha.

In [14]:
fig_pca = px.scatter(df_esol,
                     x="PCA-1",
                     y="PCA-2",
                     color='y_true',
                     title='ESOL PCA of morgan fingerprints',
                     labels={'y_true': 'Measured Solubility'},
                     width=1200,
                     height=800)

app_pca = molplotly.add_molecules(fig=fig_pca,
                                  df=df_esol.rename(columns={'y_true': 'Measured Solubility'}),
                                  smiles_col='smiles',
                                  # title_col='Compound ID',
                                  caption_cols=['Measured Solubility'],
                                  caption_transform={'Measured Solubility': lambda x: f"{x:.2f}"},
                                  color_col='Measured Solubility',
                                  show_coords=False)

app_pca.run(port=8707, height=850)


/usr/local/lib/python3.13/dist-packages/dash/dash.py:634: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.



<IPython.core.display.Javascript object>

## Clustering

Hagamos un poco de agrupación de las moléculas de ESOL, tomando prestadas funciones útiles de la excelente publicación del blog de Pat Walters sobre agrupación. [clustering](http://practicalcheminformatics.blogspot.com/2021/11/picking-highest-scoring-molecules-from.html).

In [15]:
from rdkit.ML.Cluster import Butina

def smi2fp(smi):
    fp = AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(smi), 2)
    return fp


def taylor_butina_clustering(fp_list, cutoff=0.35):
    dists = []
    nfps = len(fp_list)
    for i in range(1, nfps):
        sims = DataStructs.BulkTanimotoSimilarity(fp_list[i], fp_list[:i])
        dists.extend([1-x for x in sims])
    mol_clusters = Butina.ClusterData(dists, nfps, cutoff, isDistData=True)
    return mol_clusters


cluster_res = taylor_butina_clustering(
    [smi2fp(smi) for smi in df_esol['smiles']])
cluster_id_list = np.zeros(len(df_esol), dtype=int)
for cluster_num, cluster in enumerate(cluster_res):
    for member in cluster:
        cluster_id_list[member] = cluster_num
df_esol['cluster'] = cluster_id_list


[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerator
[14:38:08] DEPRECATION WARNING: please use MorganGenerat

Ahora hagamos un gráfico de los 10 grupos principales, veamos cómo se ven y qué tan solubles son.

In [16]:
df_cluster = df_esol.query('cluster < 15').copy().reset_index()
# sorting is needed to make the legend appear in order!
df_cluster = df_cluster.sort_values('cluster')

fig_cluster = px.strip(df_cluster,
                      y='y_true',
                      color='cluster',
                      labels={'y_true': 'Measured Solubility'},
                      width=1000,
                      height=800)

app_cluster = molplotly.add_molecules(fig=fig_cluster,
                           df=df_cluster,
                           smiles_col='smiles',
                          #  title_col='Compound ID',
                           color_col='cluster'
                           )

app_cluster.run(port=8708, height=850)


/usr/local/lib/python3.13/dist-packages/dash/dash.py:634: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.



<IPython.core.display.Javascript object>